# Glacier grids from RGI

Last step of the glacier grid pipeline (1.2 → 1.3 → 1.4). This notebook has two parts with different dependencies:

**1. Masked grids** (run *before* the regional PMB preprocessing notebooks):
combines the OGGM grids from `1.2 Create_geotiffs.ipynb` with the SVF from `1.3 Create_skyview_factor.ipynb` into one lat/lon grid per glacier: `RGI_v6/<region folder>/xr_masked_grids/<RGIId>.zarr`. The regional preprocessing notebooks (e.g. `RGI_01_02_US_CA/1.1. US_CA-prepro.ipynb`) sample the SVF at the stakes from these grids.

**2. Monthly grids** (run *after* the regional PMB preprocessing notebooks and the ERA5-Land downloads in 1.1.1/1.1.2):
for every glacier and year, builds monthly inputs (ERA5-Land climate + topography) over the whole glacier grid, so the MBM can predict PMB over the full glacier. Outputs `RGI_v6/<region folder>/monthly_parquets/` and, with `build_ds=True`, a model-ready cache in `grid_ds_cache/`. This takes a long time because of the conversion to monthly format.

## Setting up:

In [ ]:
import os

# Silence KeOps (imported indirectly via the MBM/torch stack)
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
# --- System & utilities ---
import os
import sys
import warnings
from tqdm.notebook import tqdm
import rioxarray
import pandas as pd
# Add repo root for MBM imports
sys.path.append(os.path.join(os.getcwd(), "../../"))
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from datetime import datetime
import glob
import xarray as xr
from concurrent.futures import ProcessPoolExecutor, as_completed
import glob

# --- Data science stack ---
import matplotlib.pyplot as plt

# --- Custom MBM modules ---
import massbalancemachine as mbm

# --- Warnings & autoreload (notebook) ---
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2


# --- Configuration ---
# cfg.dataPath is the data root; RGI_REGIONS, paths_multi, VOIS_CLIMATE,
# STATIC_COLS and MONTHLY_COLS come from config_TF_Europe
cfg = mbm.EuropeConfig()

from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.oggm import *
from papers.Transformer_TF.scripts.geodata import *

from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.plotting import *
from papers.Transformer_TF.scripts.models import *

# Plot styles:
mbm.utils.seed_all(cfg.seed)
mbm.plots.use_mbm_style()

print("Using seed:", cfg.seed)

if torch.cuda.is_available():
    print("CUDA is available")
    mbm.utils.free_up_cuda()
else:
    print("CUDA is NOT available")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Optional: rgi_per_region (region -> RGIIds with stakes) to restrict the grids
# to stake glaciers. Currently unused (rgi_per_region=None below).
# # Read rgis for all stakes:
# rgi_path = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_all_regions.csv")
# rgi_per_region = pd.read_csv(rgi_path, dtype={"RGI_REGION": str})

# # concatenate with HMA:
# # Read rgis for all stakes:
# rgi_path_HMA = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_HMA.csv")
# rgi_ids_HMA = pd.read_csv(rgi_path_HMA, dtype={"RGI_REGION": str})
# rgi_per_region = pd.concat([rgi_per_region, rgi_ids_HMA], ignore_index=True)

# rgi_per_region = rgi_per_region.dropna(subset=['RGIId', 'RGI_REGION']) \
#                            .groupby('RGI_REGION')['RGIId'] \
#                            .unique() \
#                            .to_dict()

## Part 1 – Masked glacier grids:

### Check resolution:
Overview of the per-glacier OGGM grids from 1.2 (resolution, extent, number of pixels), cached to `OGGM/grid_stats_cache/`. Informational: shows the spread of grid sizes that the automatic resolution choice below has to handle.

In [ ]:
GRID_STATS_CACHE_DIR = os.path.join(cfg.dataPath, "OGGM", "grid_stats_cache")
FORCE_RECOMPUTE_GRID_STATS = False  # True = rescan all zarr files instead of using the cache

regions_to_check = ["01", "11", "13"]
all_dfs = {}
for r in tqdm(regions_to_check, desc="Regions", unit="region"):
    all_dfs[r] = get_region_grid_stats(
        cfg,
        r,
        cache_dir=GRID_STATS_CACHE_DIR,
        force_recompute=FORCE_RECOMPUTE_GRID_STATS,
    )

for r, df in all_dfs.items():
    n_err = df["error"].notna().sum()
    print(f"\n=== RGI {r} ===  ({n_err} failed to open)")
    if n_err:
        print(df.loc[df["error"].notna(), ["rgi_id", "error"]].head(10))
    print(df[["dx_m", "max_extent_m",
              "n_pixels"]].describe(percentiles=[.25, .5, .75, .9, .99]))
    print("Top 5 largest by extent:")
    print(
        df.nlargest(
            5, "max_extent_m")[["rgi_id", "dx_m", "max_extent_m", "n_pixels"]])

### Masked grids:
For each glacier (`run_all_regions` → `process_one_glacier` in `scripts/dataset/create_grids.py`):
1. mask the OGGM grid to the glacier outline;
2. coarsen it depending on the glacier extent (< 500 m: native, < 5 km: 50 m, < 20 km: 100 m, else 200 m), unless `target_res_m` is set;
3. reproject to WGS84 lat/lon;
4. merge `svf`, `asvf` and `opns` (interpolated to the grid if needed) and add glacier-masked versions (`masked_svf`, ...);
5. save to `RGI_v6/<region folder>/xr_masked_grids/<RGIId>.zarr`.

Needs the SVF from `1.3 Create_skyview_factor.ipynb`: regions without OGGM grids or an `svf_nc_latlon/` folder are skipped.

In [ ]:
RUN = False  # set to True to (re)create the masked grids

if RUN:
    all_results = run_all_regions(
        cfg=cfg,
        RGI_REGIONS=RGI_REGIONS,
        rgi_per_region=None,  # dict of region -> array of RGIIds
        stake_only_regions={},  # only filter these regions
        n_workers=6,
        clear_out=False,
        target_res_m=
        None,  # None = auto-pick per glacier based on extent; set e.g. 50 to force one resolution for all
    )

In [ ]:
# Look at an example: DEM GeoTIFF (1.2), masked aspect and merged SVF for one glacier
rgi_id = "RGI60-07.00246"
region_id = "07"
# --- Paths ---
basepath = os.path.join(cfg.dataPath, "RGI_v6",
                        RGI_REGIONS[region_id]["folder"])
dem_path = os.path.join(basepath, "geotiff", f"{rgi_id}.tif")
zarr_path = os.path.join(basepath, "xr_masked_grids", f"{rgi_id}.zarr")
svf_path = os.path.join(basepath, "svf_nc_latlon", f"{rgi_id}_svf_latlon.nc")

# --- Load data ---
dem = rioxarray.open_rasterio(dem_path).squeeze()
ds = xr.open_zarr(zarr_path)
ds_svf = xr.open_dataset(svf_path)

# Handle coord naming for SVF
if "lon" not in ds_svf.coords:
    ds_svf = ds_svf.rename({"x": "lon", "y": "lat"})

# --- Figure layout ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

# 1️⃣ DEM (projected)
dem.plot(ax=axes[0], cmap="terrain")
axes[0].set_title("DEM (projected meters)")
axes[0].set_xlabel("Easting [m]")
axes[0].set_ylabel("Northing [m]")

# 2️⃣ Masked aspect (projected OGGM grid)
ds["masked_aspect"].plot(ax=axes[1])
axes[1].set_title("Masked Aspect (°)")
axes[1].set_xlabel("Longitude (°)")
axes[1].set_ylabel("Latitude (°)")

# 3️⃣ SVF (lat/lon)
ds["svf"].plot(ax=axes[2])

axes[2].set_title("Sky View Factor (lat/lon)")
axes[2].set_xlabel("Longitude (°)")
axes[2].set_ylabel("Latitude (°)")

plt.suptitle(f"{rgi_id}", fontsize=15)
plt.show()

## Part 2 – Monthly grids:
Needs the processed stake datasets of all regions (`WGMS/<region>/csv/*_wgms_dataset_all.csv`, made by the regional preprocessing notebooks) and the ERA5-Land files from 1.1.1/1.1.2.

### Load stakes and monthly stake data:
The stake data is only used here to define the glaciers with stakes and the month padding (`months_head_pad`, `months_tail_pad`) shared with model training.

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load the stake datasets of all configured regions (one dataframe per RGI region)
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

# Number of measurements per source code (CH, NOR, ...)
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
print(num_measurements)

# Regions used as training (source) and test (target) domains in cross-regional experiments
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

# True = recompute the monthly stake data from ERA5 for that code;
# False = load it from WGMS/CrossRegional/monthly_cache/<ERA5 source>/
run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code (both with the original FROM_DATE and with
# FROM_DATE shifted to Aug 1st), plus the head/tail month padding shared by all regions
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)
# Central Asia: add the RGI O2Region (subregion) of each glacier and split
# CENTRALASIA into subregions CA_12, CA_3 and CA_4
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska: same, split ALA into subregions ALA_2, ALA_4 and ALA_6. Glacier names
# that map to several RGIIds get the RGIId appended (deduplicate_glaciers)
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# Split IT_AT into Italy and Austria by glacier name
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT',
    'AT',
    'SJM',
    'ALA',
    'CA_3',
    'CA_4',
    #'CENTRALASIA', 'ALA', #'CA_12',
]

# Each source region paired with all other regions as targets
XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

# Month padding is shared by all regions, so take it from any source
months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

In [ ]:
# ERA5-Land variables used as monthly inputs
vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

# Static topographical inputs (from the masked grids)
vois_topographical = ["aspect", "slope", "svf"]

# ERA5-Land files from 1.1.2 (EU_US_CANADA) and 1.1.1 (HIGH_MOUNTAIN_ASIA)
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
    "era5_climate_data_HMA":
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
    'geopotential_data_HMA':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

In [ ]:
GRID_STATS_CACHE_DIR = os.path.join(cfg.dataPath, "OGGM", "grid_stats_cache")

# Grid stats of the OGGM grids of each region, used to compute the bounding box
# to which ERA5 is clipped for that region (saves memory and time)
regions_to_check = ["01", "06", "07", "08", "11", "13"]
FORCE_RECOMPUTE_GRID_STATS = False  # set True to rescan zarr files instead of using cache

all_dfs = {}
for r in tqdm(regions_to_check, desc="Regions", unit="region"):
    all_dfs[r] = get_region_grid_stats(
        cfg,
        r,
        cache_dir=GRID_STATS_CACHE_DIR,
        force_recompute=FORCE_RECOMPUTE_GRID_STATS,
    )

# Which ERA5-Land download each region uses
RGI_ERA5_SOURCE = {
    "01": "EU_US_CANADA",
    "06": "EU_US_CANADA",
    "07": "EU_US_CANADA",
    "08": "EU_US_CANADA",
    "11": "EU_US_CANADA",
    "13": "HMA",
}

RGI_BBOX_EMPIRICAL = {}
for r, df in all_dfs.items():
    n_err = df["error"].notna().sum()
    bbox = compute_region_bbox(df)
    RGI_BBOX_EMPIRICAL[r] = bbox
    print(f"RGI {r}: ({n_err} failed to open) -> bbox = {bbox}")

print("\nRGI_BBOX = {")
for r, bbox in RGI_BBOX_EMPIRICAL.items():
    print(f'    "{r}": {{"lat": {bbox["lat"]}, "lon": {bbox["lon"]}}},')
print("}")

In [ ]:
# Use the bounding boxes computed from the grids (alternative: fixed boxes below)
RGI_BBOX = RGI_BBOX_EMPIRICAL

In [ ]:
# Fixed, rounded bounding boxes (kept for reference)
# RGI_BBOX = {
#     "01": {
#         "lat": (52, 72),
#         "lon": (-172, -128)
#     },  # Alaska
#     "06": {
#         "lat": (62, 67),
#         "lon": (-25, -12)
#     },  # Iceland
#     "07": {
#         "lat": (74, 82),
#         "lon": (8, 36)
#     },  # Svalbard
#     "08": {
#         "lat": (56, 72),
#         "lon": (4, 32)
#     },  # Scandinavia
#     "11": {
#         "lat": (41, 49),
#         "lon": (5, 17)
#     },  # Central Europe
#     "13": {
#         "lat": (25, 45),
#         "lon": (60, 105)
#     },  # Central Asia
# }

### Run glacier grids:
For each glacier and year (`run_one_region` in `scripts/dataset/create_grids.py`): clip ERA5 to the region bounding box (cached in `ERA5Land/clipped_to_RGI/`), combine it with the masked grid and write `RGI_v6/<region folder>/monthly_parquets/<RGIId>/<RGIId>_grid_<year>.parquet`. With `build_ds=True` a model-ready dataset is also cached in `grid_ds_cache/`. Logs are written to `logs/errors/`.

Hydrological years start in October (`start_month="10"`), from 2000 to 2024.

In [ ]:
# Metadata columns kept in the monthly grid files
META_COLS = [
    'RGIId', 'POINT_ID', 'ID', 'GLWD_ID', 'N_MONTHS', 'MONTHS', 'PERIOD',
    'GLACIER'
]

#### Glaciers with stakes:
Run only for glaciers that have at least one stake in `monthly_cache`.

In [ ]:
RGI_REGIONS

In [ ]:
run_for_stake_glaciers(
    cfg=cfg,
    monthly_cache=monthly_cache,
    # monthly_cache key -> RGI region of the masked grids
    region_code_to_rgi={
        "CH": "11",
        "NOR": "08",
        "ALA": "01",
        'ISL': '06',
        'CENTRALASIA': '13',
        'SJM': '07'
    },
    RGI_REGIONS=RGI_REGIONS,
    paths=paths,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    meta_cols=META_COLS,
    years=range(2000, 2025),
    start_month="10",
    n_workers=4,
    build_ds=True,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    RGI_BBOX=RGI_BBOX,
    RGI_ERA5_SOURCE=RGI_ERA5_SOURCE,
)

#### Run specific RGIs:
Look up the RGIIds of named glaciers and run only those (useful for tests and figures).

In [ ]:
# RGIId(s) of a glacier by name
glacier = 'aletsch'
df_CH = monthly_cache['CH']['data_monthly']
rgi_gl = df_CH[df_CH.GLACIER == glacier].RGIId.unique()

print(f'{glacier}: {rgi_gl}')

In [ ]:
glacier = 'Aalfotbreen'
region = 'NOR'
df_reg = monthly_cache[region]['data_monthly']
rgi_gl = df_reg[df_reg.GLACIER == glacier].RGIId.unique()

print(f'{glacier}: {rgi_gl}')

In [ ]:
# resolve glacier names -> RGI IDs from monthly_cache
# aletsch_ids = monthly_cache["CH"]["data_monthly"].query(
#     "GLACIER == 'aletsch'")["RGIId"].unique().tolist()
nigards_ids = monthly_cache["NOR"]["data_monthly"].query(
    "GLACIER == 'Nigardsbreen'")["RGIId"].unique().tolist()
# enga_ids = monthly_cache["NOR"]["data_monthly"].query(
#     "GLACIER == 'Engabreen'")["RGIId"].unique().tolist()

# print(f"Aletsch: {aletsch_ids}")
# print(f"Nigardsbreen: {nigards_ids}")
# print(f"Engabreen: {enga_ids}")

# glacier_rgi_ids: RGI region -> list of RGIIds to process
run_for_glaciers(cfg=cfg,
                 glacier_rgi_ids={
                    #  "11": aletsch_ids,
                     "08": ['RGI60-08.02666'],
                 },
                 RGI_REGIONS=RGI_REGIONS,
                 paths=paths,
                 vois_climate=vois_climate,
                 vois_topographical=vois_topographical,
                 meta_cols=META_COLS,
                 years=range(2000, 2025),
                 start_month="10",
                 n_workers=4,
                 build_ds=True,
                 monthly_cols=MONTHLY_COLS,
                 static_cols=STATIC_COLS,
                 months_head_pad=months_head_pad,
                 months_tail_pad=months_tail_pad,
                 RGI_BBOX=RGI_BBOX,
                 RGI_ERA5_SOURCE=RGI_ERA5_SOURCE)

#### Run all RGIs:
Run for all glaciers of the regions (larger than 1 km² by default). This takes very long.

In [ ]:
# NOTE: RGI_REGIONS_TEST is not defined anywhere; replace it before setting RUN = True
RUN = False
max_workers = max(1, (os.cpu_count() or 4) - 1)

if RUN:
    all_results = run_all_glacier_grids(
        cfg=cfg,
        RGI_REGIONS=RGI_REGIONS_TEST,
        rgi_per_region=None,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        meta_cols=META_COLS,
        years=range(2000, 2025),
        start_month="10",
        n_workers=10,
        force_recompute=False,
        build_ds=True,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        RGI_BBOX=RGI_BBOX,
        RGI_ERA5_SOURCE=RGI_ERA5_SOURCE,
    )

In [ ]:
META_COLS = [
    'RGIId', 'POINT_ID', 'ID', 'GLWD_ID', 'N_MONTHS', 'MONTHS', 'PERIOD',
    'GLACIER'
]

RUN = True
max_workers = max(1, (os.cpu_count() or 4) - 1)
"""
    '07': {
        'name': 'Svalbard',
        'folder': 'RGI_07_Svalbard',
        'file': '07_rgi60_Svalbard.shp',
        'code': 'SJM',
        'era5_source': 'EU_US_CANADA'
    },"""

# Subset of regions to run. NOTE: this overwrites RGI_REGIONS from the config
# for the rest of the session.
RGI_REGIONS = {
    '11': {
        'name': 'CentralEurope',
        'folder': 'RGI_11_CentralEurope',
        'file': '11_rgi60_CentralEurope.shp',
        'code': 'CEU',
        'subregions': ['France', 'Switzerland', 'Italy_Austria'],
        'subregions_codes': ['FR', 'CH', 'IT_AT'],
        'era5_source': 'EU_US_CANADA'
    },
    '06': {
        'name': 'Iceland',
        'folder': 'RGI_06_Iceland',
        'file': '06_rgi60_Iceland.shp',
        'code': 'ISL',
        'era5_source': 'EU_US_CANADA'
    },
    '08': {
        'name': 'Scandinavia',
        'folder': 'RGI_08_Scandinavia',
        'file': '08_rgi60_Scandinavia.shp',
        'code': 'SCA',
        'subregions': ['Norway'],
        'subregions_codes': ['NOR'],
        'era5_source': 'EU_US_CANADA'
    },
}

if RUN:
    all_results = run_all_glacier_grids(
        cfg=cfg,
        RGI_REGIONS=RGI_REGIONS,
        rgi_per_region=None,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        meta_cols=META_COLS,
        years=range(2000, 2025),
        start_month="10",
        n_workers=10,
        force_recompute=False,
        build_ds=True,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        RGI_BBOX=RGI_BBOX,
        RGI_ERA5_SOURCE=RGI_ERA5_SOURCE,
    )